# Task4 海龟交易策略回测 Notebook

本 Notebook 用白盒方式展示海龟交易策略的核心流程，包括：价格通道计算、ATR 计算、突破信号生成、止损规则以及策略回测。

## 1. 核心概念

- **高低点通道**：过去若干日的最高价和最低价区间。
- **ATR**：平均真实波幅，用于衡量波动强度。
- **止损条件**：本次采用 `2 × ATR` 的波动性止损。
- **累计回报、最大回撤、夏普比率**：用于评价策略收益与风险。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

base_dir = Path.cwd().resolve().parent
sys.path.insert(0, str(base_dir))

from turtle_utils import TurtleParams, apply_turtle_strategy, load_price_data, summarize_strategy

raw_dir = base_dir / 'raw_data'
processed_dir = base_dir / 'processed_data'
figures_dir = base_dir / 'figures'
default_params = TurtleParams(entry_window=20, exit_window=10, atr_window=20, stop_atr_multiplier=2.0)
metrics_df = pd.read_csv(processed_dir / 'turtle_strategy_metrics.csv')
metrics_df

## 2. 以中国巨石为例计算海龟策略

下面展示从原始股价数据出发，计算通道、ATR、止损线和交易信号。

In [ ]:
jushi = load_price_data(raw_dir / 'china-jushi_600176.csv')
jushi_strategy = apply_turtle_strategy(jushi, default_params.entry_window, default_params.exit_window, default_params.atr_window, default_params.stop_atr_multiplier)
jushi_strategy[['date', 'close', 'entry_high', 'exit_low', 'atr', 'buy_signal', 'sell_signal', 'sell_reason', 'position']].tail(15)

In [ ]:
jushi_summary = summarize_strategy(jushi_strategy, '中国巨石（600176）', default_params)
pd.Series(jushi_summary)

## 3. 图表展示

以下图表展示了默认参数 `20/10/ATR20/2N` 下的交易信号图和净值曲线。

In [ ]:
display(Image(filename=str(figures_dir / 'china_jushi_turtle_20_10_signals.png')))
display(Image(filename=str(figures_dir / 'china_jushi_turtle_20_10_equity.png')))
display(Image(filename=str(figures_dir / 'zhongji_xuchuang_turtle_20_10_signals.png')))
display(Image(filename=str(figures_dir / 'zhongji_xuchuang_turtle_20_10_equity.png')))

## 4. 不同通道参数比较

为了比较策略适应性，本次另外测试了 `30/10` 和 `55/20` 两组通道长度。

In [ ]:
metrics_df[['stock', 'entry_window', 'exit_window', 'atr_window', 'stop_atr_multiplier', 'strategy_cumulative_return', 'max_drawdown', 'sharpe_ratio']]

In [ ]:
display(Image(filename=str(figures_dir / 'turtle_strategy_metric_comparison.png')))

## 5. 观察总结

1. 海龟策略属于趋势突破策略，适合趋势明显的阶段。
2. ATR 有助于让止损规则随波动变化，而不是机械使用固定点位。
3. 更长的通道会减少交易次数，但也可能错过部分行情；更短的通道则更灵敏，但更容易受到噪声影响。